# Re_390

## 1.Import Libreries

In [ ]:
import numpy as np
import pandas as pd
import scipy.io
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold

import warnings
warnings.filterwarnings("ignore")

## 2.Import dataset

In [ ]:
# Load .mat file
data = scipy.io.loadmat('data_properties_390.mat')

In [ ]:
F1=np.reshape(data['f1'][:,:],[257*256])
F2=np.reshape(data['f2'][:,:],[257*256])
F3=np.reshape(data['f3'][:,:],[257*256])
F4=np.reshape(data['f4'][:,:],[257*256])
F5=np.reshape(data['f5'][:,:],[257*256])
F6=np.reshape(data['f6'][:,:],[257*256])
F7=np.reshape(data['f7'][:,:],[257*256])
F8=np.reshape(data['f8'][:,:],[257*256])
F9=np.reshape(data['f9'][:,:],[257*256])
F10=np.reshape(data['f10'][:,:],[257*256])
index=np.arange(1,(257*256)+1)
col=['wall_distance' , 'Uave' ,'Vave' ,'Wave', 'TKE' , 'strain_rate' ,'dissipation_rate'
     ,'Cross_Correlataion', 'eddy_Viscosity' , 'Reynolds_Stress']
df=pd.DataFrame(list(zip(F1,F2,F3,F4,F5,F6,F7,F8,F9,F10)) , index=index , columns=col)

In [ ]:
df

## 3.Normalization with Min_Max

In [ ]:
X=df.iloc[:,:-2]
Y=df.iloc[:,-2]

In [ ]:
x_train , x_test , y_train , y_test = train_test_split(X , Y , train_size=.20 , random_state=4040)

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(x_train)
X_test_sc = scaler.transform(x_test)


## 4.Feature selection with random forest

In [ ]:
%%time 
# Create a Random Forest Regressor
rf = RandomForestRegressor(n_estimators=50, random_state=2020)


rf.fit(x_train , y_train) 

# Get feature importances
feature_importances = rf.feature_importances_

# Create a DataFrame to store feature importances
importance_df = pd.DataFrame({'Feature': df.columns[:-2], 'Importance': feature_importances})

# Sort the features by importance in descending order
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Print or visualize the feature importances
print(importance_df)


## 6.Grid search

In [ ]:
def ann_model(number_of_hidden_layers=1,
              number_of_neurons=50, optimizer='adam'):
    model = keras.models.Sequential()
    model.add(keras.layers.InputLayer(input_shape=[8]))
    for hidden_layer in range(number_of_hidden_layers):
        model.add(keras.layers.Dense(number_of_neurons, activation="selu"))
    model.add(keras.layers.Dense(1))
    model.compile(loss="mse", optimizer=optimizer)  # Use the provided optimizer
    return model

In [ ]:
keras_sk_reg = keras.wrappers.scikit_learn.KerasRegressor(build_fn=ann_model)

In [ ]:
param_grid = {
    "number_of_hidden_layers": [1, 2, 3, 4, 5, 6],
    "number_of_neurons": [10, 20, 30, 40, 50],
    "optimizer": ['adam', 'SGD', 'SGD with momentum', 'rmsprop']
}


In [ ]:
keras_sk_reg_gs = GridSearchCV(keras_sk_reg, param_grid , cv=5)

In [ ]:
model=keras_sk_reg_gs.fit(x_train_sc, y_train, epochs=100,verbose=0,validation_split=.2 ,
                    callbacks=[keras.callbacks.EarlyStopping(patience=16)])

## 7.Train Model

In [ ]:
Model=keras.models.Sequential([
    keras.layers.Dense(75 , activation='selu'),
    keras.layers.Dense(65 ,activation='selu'),
    keras.layers.Dense(55,activation='selu'),
    keras.layers.Dense(1)
])

In [ ]:
opt=keras.optimizers.Adam(learning_rate=.001)
Model.compile(loss="mean_squared_error",
              optimizer=opt,
              metrics=["mean_squared_error"])

In [ ]:
hist=Model.fit(x_train_sc , y_train ,verbose=0 , validation_split=.20 , epochs=250)

## 8.K_fold Cross validation

In [ ]:

kfold = KFold(n_splits=5, shuffle=True)

# Fit the model
Model.fit(X_train, y_train, epochs=10, validation_data=kfold, verbose=0)


## 9.Evaluation

In [ ]:
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
print("Mean Squared Error:", mse)
print('*************************************************')
print("Root Mean Squared Error:", rmse)
print('*************************************************')
print("Mean Absolute Error:", mae)
print('*************************************************')
print("R-squared:", r2)


## 12.Save Model

In [ ]:
model.save('Training_model_reynolds_390.h5')

# Re 180

In [ ]:
# Load .mat file
data_180 = scipy.io.loadmat('data_properties_180.mat')

In [ ]:
E1=np.reshape(data['f1'][:,:],[129*128])
E2=np.reshape(data['f2'][:,:],[129*128])
E3=np.reshape(data['f3'][:,:],[129*128])
E4=np.reshape(data['f4'][:,:],[129*128])
E5=np.reshape(data['f5'][:,:],[129*128])
E6=np.reshape(data['f6'][:,:],[129*128])
E7=np.reshape(data['f7'][:,:],[129*128])
E8=np.reshape(data['f8'][:,:],[129*128])
E9=np.reshape(data['f9'][:,:],[129*128])
E10=np.reshape(data['f10'][:,:],[129*128])
index=np.arange(1,(129*128)+1)
col=['wall_distance' , 'Uave' ,'Vave' ,'Wave', 'TKE' , 'strain_rate' ,'dissipation_rate'
     ,'Cross_Correlataion', 'eddy_Viscosity' , 'Reynolds_Stress']
df2=pd.DataFrame(list(zip(E1,E2,E3,E4,E5,E6,E7,E8,E9,E10)) , index=index , columns=col)

In [ ]:
df2

In [ ]:
X2=df2.iloc[:,:-2]
Y2=df2.iloc[:,-2]

In [ ]:

X_train_sc2 = scaler.transform(X2)

In [ ]:
y_pred2=Model.predict(X_train_sc2)

In [ ]:
mse2 = mean_squared_error(Y2, y_pred2)
rmse2 = np.sqrt(mse2)
mae2 = mean_absolute_error(Y2, y_pred2)
print("Mean Squared Error:", mse2)
print('*************************************************')
print("Root Mean Squared Error:", rmse2)
print('*************************************************')
print("Mean Absolute Error:", mae2)
print('*************************************************')

# Re 590

In [ ]:
# Load .mat file
data_180 = scipy.io.loadmat('data_properties_590.mat')

In [ ]:
G1=np.reshape(data['f1'][:,:],[257*384])
G2=np.reshape(data['f2'][:,:],[257*384])
G3=np.reshape(data['f3'][:,:],[257*384])
G4=np.reshape(data['f4'][:,:],[257*384])
G5=np.reshape(data['f5'][:,:],[257*384])
G6=np.reshape(data['f6'][:,:],[257*384])
G7=np.reshape(data['f7'][:,:],[257*384])
G8=np.reshape(data['f8'][:,:],[257*384])
G9=np.reshape(data['f9'][:,:],[257*384])
G10=np.reshape(data['f10'][:,:],[257*384])
index=np.arange(1,(257*384)+1)
col=['wall_distance' , 'Uave' ,'Vave' ,'Wave', 'TKE' , 'strain_rate' ,'dissipation_rate'
     ,'Cross_Correlataion', 'eddy_Viscosity' , 'Reynolds_Stress']
df3=pd.DataFrame(list(zip(G1,G2,G3,G4,G5,G6,G7,G8,G9,G10)) , index=index , columns=col)

In [ ]:
df3

In [ ]:
X2=df3.iloc[:,:-2]
Y2=df3.iloc[:,-2]

In [ ]:

X_train_sc3 = scaler.transform(X3)

In [ ]:
y_pred3=Model.predict(X_train_sc3)

In [ ]:
mse3 = mean_squared_error(Y2, y_pred3)
rmse3 = np.sqrt(mse3)
mae3 = mean_absolute_error(Y2, y_pred3)
print("Mean Squared Error:", mse3)
print('*************************************************')
print("Root Mean Squared Error:", rmse3)
print('*************************************************')
print("Mean Absolute Error:", mae3)
print('*************************************************')